In [21]:
import boto3
import polars as pl
import time
from decimal import Decimal
from typing import Any, Dict, List, Sequence
from boto3.dynamodb.conditions import Attr

# DynamoDB client
dynamo = boto3.resource("dynamodb", region_name="ap-northeast-2")
tbl = dynamo.Table('event')

In [22]:
def _to_py(v):
    """DynamoDB Decimal/중첩 정규화"""
    if isinstance(v, Decimal):
        return int(v) if v == v.to_integral_value() else float(v)
    if isinstance(v, dict):
        return {k: _to_py(x) for k, x in v.items()}
    if isinstance(v, list):
        return [_to_py(x) for x in v]
    return v

def _ensure_keys(d: Dict[str, Any], keys: Sequence[str]) -> Dict[str, Any]:
    for k in keys:
        if k not in d:
            d[k] = None
    return d

In [23]:
def fetch_events_last_ndays(days: int = 7, tbl=None) -> pl.DataFrame:
    if tbl is None:
        raise ValueError("tbl (boto3.resource('dynamodb').Table) 을 인자로 넘겨주세요.")

    end_ms = int(time.time() * 1000)
    start_ms = end_ms - days * 24 * 60 * 60 * 1000

    # 예약어 우회: timestamp -> #ts
    expr_attr_names = {"#ts": "timestamp"}
    projection_expression = "member_id, event_type, target_type, target_id, #ts"

    filter_expression = Attr("timestamp").between(start_ms, end_ms)

    items: List[Dict[str, Any]] = []
    last_evaluated_key: Dict[str, Any] | None = None

    while True:
        scan_kwargs = {
            "FilterExpression": filter_expression,
            "ExpressionAttributeNames": expr_attr_names,
            "ProjectionExpression": projection_expression,
            "Limit": 1000,
        }
        if last_evaluated_key:
            scan_kwargs["ExclusiveStartKey"] = last_evaluated_key

        resp = tbl.scan(**scan_kwargs)
        page_items = resp.get("Items", [])
        if page_items:
            items.extend(page_items)

        last_evaluated_key = resp.get("LastEvaluatedKey")
        if not last_evaluated_key:
            break

    # 빈 결과: 스키마만 가진 DF 반환
    if not items:
        return pl.DataFrame(
            schema={
                "member_id": pl.Int64,
                "event_type": pl.Utf8,
                "target_type": pl.Utf8,
                "target_id": pl.Utf8,
                "timestamp": pl.Int64,
                "datetime_utc": pl.Datetime("ms"),
            }
        )

    wanted = ["member_id", "event_type", "target_type", "target_id", "timestamp"]
    items_norm = [_ensure_keys(_to_py(it), wanted) for it in items]

    # 스키마 고정 생성
    df = pl.from_dicts(
        items_norm,
        schema={
            "member_id": pl.Int64,
            "event_type": pl.Utf8,
            "target_type": pl.Utf8,
            "target_id": pl.Utf8,
            "timestamp": pl.Int64,  # ms epoch
        },
    ).with_columns(
        pl.col("member_id").cast(pl.Int64, strict=False),
        pl.col("event_type").cast(pl.Utf8, strict=False),
        pl.col("target_type").cast(pl.Utf8, strict=False),
        pl.col("target_id").cast(pl.Utf8, strict=False),
        pl.col("timestamp").cast(pl.Int64, strict=False),
    )

    df = df.with_columns(
        pl.from_epoch(pl.col("timestamp"), "ms")
          .dt.replace_time_zone("UTC")
          .alias("datetime_utc")
    )

    # 최종 범위 안전 필터 
    df = df.filter(
        pl.col("timestamp").is_not_null()
        & (pl.col("timestamp") >= start_ms)
        & (pl.col("timestamp") <= end_ms)
    )

    return df

In [29]:
def classify_users_quantile(df: pl.DataFrame) -> pl.DataFrame:
    now_ms = int(time.time() * 1000)

    # 유저별 집계
    user_stats = (
        df.group_by("member_id")
          .agg([
              pl.col("timestamp").max().alias("last_event_ts"),
              pl.len().alias("event_count"),
              pl.col("event_type").n_unique().alias("event_diversity"),
          ])
          .with_columns(
              ((now_ms - pl.col("last_event_ts")) / (1000 * 60 * 60 * 24)).alias("recency_days")
          )
    )

    if user_stats.is_empty():
        return user_stats.with_columns(pl.lit("cold").alias("user_segment"))

    # 분위수(이벤트 수 기준) — 데이터 분포에 따라 자동으로 경계 설정
    q25 = user_stats["event_count"].quantile(0.25, "nearest")
    q50 = user_stats["event_count"].quantile(0.50, "nearest")
    q75 = user_stats["event_count"].quantile(0.75, "nearest")

    # 분류 규칙 (예)
    # hot: 최근 1일 내 활동 & 이벤트수 상위 25% & 다양성 ≥3
    # warm: 최근 3일 내 활동 & 이벤트수 중간 이상 & 다양성 ≥2
    # else: cold
    return (
        user_stats.with_columns(
            pl.when(
                (pl.col("recency_days") <= 1)
                & (pl.col("event_count") >= q75)
                & (pl.col("event_diversity") >= 3)
            ).then(pl.lit("hot"))
             .when(
                (pl.col("recency_days") <= 3)
                & (pl.col("event_count") >= q50)
                & (pl.col("event_diversity") >= 2)
            ).then(pl.lit("warm"))
             .otherwise(pl.lit("cold"))
             .alias("user_segment")
        )
    )

In [30]:
df = fetch_events_last_ndays(7, tbl)

In [31]:
segments = classify_users_quantile(df)

In [37]:
df

member_id,event_type,target_type,target_id,timestamp,datetime_utc
i64,str,str,str,i64,"datetime[ms, UTC]"
187,"""r_imp""","""article""","""4Rbpk2DMYh83bLwIEUgQ1nI7L2h""",1755335562349,2025-08-16 09:12:42.349 UTC
187,"""r_imp""","""article""","""4Rbpk2DMYh83bLwIEUgQ1nI7L2h""",1755335566017,2025-08-16 09:12:46.017 UTC
187,"""r_imp""","""article""","""7aS23vKt0IqymYI2vgFUbbYWmrh""",1755335566351,2025-08-16 09:12:46.351 UTC
187,"""r_imp""","""article""","""cesKf5YMunGnBPP5Jcy8mXd8n6z""",1755335573434,2025-08-16 09:12:53.434 UTC
187,"""r_imp""","""article""","""cesKf5YMunGnBPP5Jcy8mXd8n6z""",1755335580997,2025-08-16 09:13:00.997 UTC
…,…,…,…,…,…
186,"""r_imp""","""article""","""1vYXONifJS5OlvrMD0DxEtkZLIZ""",1755335383342,2025-08-16 09:09:43.342 UTC
186,"""r_imp""","""article""","""1vYXONifJS5OlvrMD0DxEtkZLIZ""",1755335449709,2025-08-16 09:10:49.709 UTC
186,"""r_imp""","""article""","""1vYXONifJS5OlvrMD0DxEtkZLIZ""",1755335463477,2025-08-16 09:11:03.477 UTC
